# Experiment Tracking with MLflow

> 📘 **Python Mastery** · Module 18 — MLOps · Lesson 3/6

Fifty training runs later, which configuration actually won? This lesson gives every run a memory — parameters, metrics, artifacts and models recorded automatically with MLflow on a local file store, plus a twenty-line from-scratch tracker that shows exactly what MLflow does for you.

## 🎯 Learning Objectives

- Explain what an experiment-tracking system records and why a spreadsheet cannot keep up
- Point MLflow at a **local file-based tracking URI** (`mlruns/`) so tracking works fully offline
- Log params, metrics, tags and artifacts inside `with mlflow.start_run():` blocks
- Run several configurations as named runs in one experiment and compare their metrics
- Query history with `mlflow.search_runs`, sort by any metric and reload the winner's artifact
- Build a minimal JSON tracker that mirrors the semantics of a real tracking server

## 1. The Archaeology Problem

Three weeks of tuning leave behind `final_final_v3_REAL.pkl`, a phone gallery of metric
screenshots, and no reliable memory of which learning rate produced which score. An
experiment tracker is an **automatic lab notebook**: every training run files one record
containing everything you would need to explain — or rerun — it.

| A run record holds | Why you will need it |
|---|---|
| Identity (experiment, run name, timestamps) | "Which attempt was this?" |
| Parameters (`C=0.5`, `seed=42`, `dataset_tag`) | "What exactly did we try?" |
| Metrics (`accuracy`, `recall_fraud`) logged per step | "How good was it — and did it improve during training?" |
| Artifacts (`config.json`, `model.joblib`, plots) | "Can I get the actual model back?" |
| Source lineage (git SHA, user) | "Whose code produced this?" |

Spreadsheets fail because humans forget to fill rows; trackers fail *never* — they are
called by the training code itself.

## 2. MLflow in One Breath

MLflow organises the world as **experiments → runs**. An experiment groups related attempts
(one project, e.g. `churn-model`); each run is one execution with its params, metrics and
artifacts. Everything lands in a **tracking store**, and for local work that store is simply
a folder of small files — no server, no database, no internet.

**Syntax:** aim MLflow at a folder and it behaves identically on every machine:

```python
import mlflow
from pathlib import Path

TRACKING_URI = (Path("mlruns").resolve()).as_uri()   # file:... URI -> ./mlruns/
mlflow.set_tracking_uri(TRACKING_URI)                # offline, portable, inspectable
mlflow.set_experiment("churn-model")                 # groups the runs that follow
```

Install once, in your terminal — never inside a code cell:

```bash
pip install mlflow
```

> 🔍 **Under the Hood:** a file-based tracking URI makes MLflow write ordinary directories:
> `mlruns/<experiment_id>/<run_id>/params/<name>`, `.../metrics/<name>` (one line per step:
> timestamp, value, step), and `.../artifacts/`. Because the store is *just files*, you can
> grep it, zip it, commit it or point teammates at a shared drive — and `mlflow ui`
> renders the same store in a browser. The URI scheme (`file:`) is why nothing here needs
> a network.

In [ ]:
# Verify the setup: MLflow aimed at a local folder
from pathlib import Path

import mlflow

TRACKING_URI = (Path("mlruns").resolve()).as_uri()
mlflow.set_tracking_uri(TRACKING_URI)

exp = mlflow.set_experiment("setup-check")
print("tracking uri :", mlflow.get_tracking_uri())
print("experiment   :", exp.name, "(id", exp.experiment_id, ")")
print("artifact root:", Path("mlruns").resolve())

## 3. Your First Tracked Run

A run is opened with `mlflow.start_run()` and — crucially — closed by the context manager.
Inside the block, three workhorse calls do most of the job:

| Call | Records | Example |
|---|---|---|
| `mlflow.log_param(k, v)` / `log_params(dict)` | Inputs you CHOSE | `C`, `seed`, `dataset_tag` |
| `mlflow.log_metric(k, v)` / `log_metrics(dict)` | Results you MEASURED | `accuracy`, `recall` |
| `mlflow.set_tag(k, v)` | Free-form labels for filtering | `author=sarah`, `baseline=yes` |

**Rule of thumb:** params before training, metrics after evaluation, tags whenever.

In [ ]:
# A complete, honest baseline run
import tempfile
from pathlib import Path

import mlflow
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                          random_state=42, stratify=y)

with mlflow.start_run(run_name="logreg-baseline") as run:
    mlflow.log_params({"model": "LogisticRegression", "C": 1.0,
                       "seed": 42, "dataset_tag": "breast_cancer@2026-08"})
    mlflow.set_tags({"author": "sarah", "stage": "baseline"})

    pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
    pipe.fit(X_tr, y_tr)
    acc = pipe.score(X_te, y_te)
    mlflow.log_metric("accuracy", round(acc, 4))

    # a tiny human-readable artifact alongside the numbers
    report = Path(tempfile.mkdtemp()) / "test_metrics.json"
    report.write_text('{"test_accuracy": ' + f"{acc:.4f}" + '}')
    mlflow.log_artifact(str(report), artifact_path="reports")

print("run id      :", run.info.run_id)
print("status      :", run.info.status)          # 'FINISHED' thanks to the context manager
print("accuracy    :", round(acc, 4))

## 4. Comparing Runs: the Whole Point

Tracking earns its keep the moment configurations multiply. Give each attempt a meaningful
`run_name`, log identical metric names across runs, and the comparison becomes a query —
not an afternoon of scrolling through notebook outputs.

In [ ]:
# Sweep regularisation strength: three runs, one experiment
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.datasets import load_breast_cancer

mlflow.set_experiment("c-sweep")

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                          random_state=42, stratify=y)

for c_value in (0.01, 0.1, 1.0):
    with mlflow.start_run(run_name=f"logreg_C={c_value}"):
        pipe = make_pipeline(StandardScaler(),
                             LogisticRegression(C=c_value, max_iter=2000))
        pipe.fit(X_tr, y_tr)
        mlflow.log_params({"C": c_value, "model": "LogisticRegression"})
        mlflow.log_metric("accuracy", round(pipe.score(X_te, y_te), 4))

print("three runs filed under experiment 'c-sweep'")

In [ ]:
# Ask the tracker: which run won?
import mlflow

exp = mlflow.get_experiment_by_name("c-sweep")

results = mlflow.search_runs(
    experiment_ids=[exp.experiment_id],
    order_by=["metrics.accuracy DESC"],
    max_results=3,
)

league = results[["tags.mlflow.runName", "params.C", "metrics.accuracy"]]
print(league.to_string(index=False))
print("\nWinner:", league.iloc[0]["tags.mlflow.runName"],
      "at", league.iloc[0]["metrics.accuracy"])

## 5. Artifacts: Getting the Actual Model Back

Metrics tell you *who won*; artifacts let you *keep playing*. Dump the fitted pipeline with
joblib, attach it to the run with `log_artifact`, and the winning model becomes retrievable
by run ID alone — no shared filesystem, no "check my machine".

**Syntax:**

```python
mlflow.log_artifact(local_path, artifact_path="models")   # one file...
mlflow.log_artifacts(local_dir, artifact_path="models")   # ...or a whole folder
```

In [ ]:
# Attach the model to the winning run, then pull it back out by run ID
import json
import tempfile
from pathlib import Path

import joblib
import mlflow
from mlflow.tracking import MlflowClient
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

X, y = load_breast_cancer(return_X_y=True)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                          random_state=42, stratify=y)
pipe = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=2000))
pipe.fit(X_tr, y_tr)

with mlflow.start_run(run_name="with-artifact") as run:
    mlflow.log_metric("accuracy", round(pipe.score(X_te, y_te), 4))
    model_file = Path(tempfile.mkdtemp()) / "model.joblib"
    joblib.dump(pipe, model_file)
    mlflow.log_artifact(str(model_file), artifact_path="models")

client = MlflowClient()
print("artifacts stored:", [f.path for f in client.list_artifacts(run.info.run_id)])

local_copy = client.download_artifacts(run.info.run_id, "models/model.joblib",
                                       tempfile.mkdtemp())
revived = joblib.load(local_copy)
assert (revived.predict(X_te) == pipe.predict(X_te)).all()
print("downloaded artifact reproduces the original predictions exactly.")

## 6. A Twenty-Line Tracker From Scratch

MLflow's file store is not magic — it is the discipline below, automated. Building the
smallest possible tracker once teaches you precisely what a tracking server buys you, and
it works in environments where installing anything is forbidden.

In [ ]:
# runs/ tracker: one JSON file per run, newest wins queries
import json
from datetime import datetime, timezone
from pathlib import Path


class JsonTracker:
    def __init__(self, base="sample_data/tracking"):
        self.base = Path(base)
        self.base.mkdir(parents=True, exist_ok=True)

    def log_run(self, name, params, metrics):
        record = {
            "run_name": name,
            "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "params": params,
            "metrics": metrics,
        }
        path = self.base / f"{name}.json"
        path.write_text(json.dumps(record, indent=2))
        return path

    def best(self, metric):
        runs = [json.loads(p.read_text())
                for p in self.base.glob("*.json")]
        return max(runs, key=lambda r: r["metrics"][metric])


tracker = JsonTracker()
tracker.log_run("lr_c0.01", {"C": 0.01}, {"accuracy": 0.951})
tracker.log_run("lr_c0.1",  {"C": 0.1},  {"accuracy": 0.965})
tracker.log_run("lr_c1.0",  {"C": 1.0},  {"accuracy": 0.958})

winner = tracker.best("accuracy")
print("files:", sorted(p.name for p in Path('sample_data/tracking').glob('*.json')))
print(f"best by accuracy: {winner['run_name']} ({winner['metrics']['accuracy']})")

## 7. The Model Registry, Conceptually

A tracker remembers *attempts*; a registry governs *the chosen few*. One model name holds
many versions, and each version sits in a stage:

| Stage | Meaning | Who moves it |
|---|---|---|
| `None` | freshly registered candidate | automatic at registration |
| `Staging` | promoted for pre-production testing | a merge/gate, not a mood |
| `Production` | what applications call | the promotion gate (Lesson 06) |
| `Archived` | retired but auditable | rollback or deprecation |

Promotion should be a **gate crossing, not a meeting**: new version beats incumbent on the
agreed metric → advance; otherwise stay. The file-based tracker has no registry UI, so the
honest offline equivalent is a tiny pointer file — which is exactly what registries do,
with permissions and audit trails bolted on.

In [ ]:
# Registry thinking, offline edition: versions + a stage pointer
import json
from pathlib import Path

reg = Path("sample_data/registry/churn-model")
(reg / "1").mkdir(parents=True, exist_ok=True)
(reg / "2").mkdir(parents=True, exist_ok=True)
(reg / "1" / "metrics.json").write_text('{"accuracy": 0.941}')
(reg / "2" / "metrics.json").write_text('{"accuracy": 0.962}')
(reg / "2" / "STAGE").write_text("Production")     # pointer IS the deployment truth
(reg / "1" / "STAGE").write_text("Archived")


def production_version(name: str, root="sample_data/registry") -> str:
    root = Path(root) / name
    for version_dir in sorted(root.iterdir()):
        if (version_dir / "STAGE").read_text() == "Production":
            return version_dir.name
    raise LookupError(f"no Production version for {name}")


print("serving churn-model version:", production_version("churn-model"))
print("promotion rule held: v2 beat v1 on accuracy, so the pointer moved.")

## ⚠️ Common Mistakes & Gotchas

| Mistake | Problem | Fix |
|---|---|---|
| Opening `start_run()` without closing it | The run stays ACTIVE; later logs land inside it | Always use `with mlflow.start_run():` |
| Logging metrics after the run ended | `MlflowException: ... must be active` — data lost | Log inside the block, or reopen with the run ID |
| Different param names per attempt (`lr` vs `learning_rate`) | Comparison queries fall apart | Agree a param schema once; rename deliberately |
| Relative `mlruns/` paths on shared machines | Each process creates its OWN folder wherever it started | Resolve to an absolute URI: `Path("mlruns").resolve().as_uri()` |
| Treating the tracker as the backup | File stores get deleted with the laptop | Push artifacts to git/DVC/storage; tracker is the index, not the vault |
| Logging metrics inside the epoch loop as fresh params | Hundreds of meaningless params bury the useful ones | Loop values are metric *steps*: `log_metric(metric, value, step=epoch)` |
| Naming runs `run1`, `run2`, `final` | Search becomes archaeology again | Name by intent: `logreg_C0.1_class-balanced` |

## 💡 Best Practices & Pro Tips

- **One experiment per question, one run per attempt.** "Does calibrating help?" is an
  experiment; every hyperparameter combo inside it is a run.
- **Log the dataset identity, not just the code**: a `dataset_tag` or content hash param
  turns "which data?" into a filter instead of an interrogation.
- **Log a small human-readable artifact** (`metrics.json`, a confusion-matrix PNG). Six
  months later the artifact is what colleagues actually open.
- **Query, don't scroll**: learn `search_runs(order_by=[...])` early — leaderboards beat
  memory every time.
- **AI-engineering relevance:** LLM experiments deserve the same ledger. Track prompts
  (params), temperature/top_p (params), win-rate and token cost (metrics), sample outputs
  (artifacts). "Which prompt version won last Tuesday?" should be one query away.
- **Keep the from-scratch tracker in your head.** When a platform is down or forbidden,
  the pattern — identity + params + metrics + artifacts, one folder per run — still ships.

## 📌 Summary

| Tool / Call | What it does | Example |
|---|---|---|
| `set_tracking_uri(uri)` | Aim MLflow at a store | `Path("mlruns").resolve().as_uri()` |
| `set_experiment(name)` | Group related runs | `set_experiment("churn-model")` |
| `start_run()` (context manager) | Open/close one attempt | `with mlflow.start_run(run_name=...):` |
| `log_params` / `log_metrics` | Record choices and results | `log_metric("accuracy", 0.962)` |
| `log_artifact(path)` | Attach files to the run | models, configs, plots |
| `search_runs(..., order_by=...)` | Query history | `order_by=["metrics.accuracy DESC"]` |
| `MlflowClient.download_artifacts` | Retrieve a run's files | reload the winning model |
| Registry (concept) | Version + stage chosen models | staging -> production via a gate |

Key takeaways:

- A tracker converts tuning from folklore into a queryable ledger of `(params -> metrics)`.
- A file-based tracking URI keeps everything offline: `mlruns/` is the whole system.
- Context managers close runs; unclosed runs silently swallow later logs.
- Metrics name the scoreboard consistently, artifacts carry the goods, queries pick winners.

## 🔗 Next Lesson

Up next: **[04_Model_Serving_FastAPI](../04_Model_Serving_FastAPI/notes.ipynb)** — the winning model leaves the notebook: a validated HTTP API built and tested without ever starting a server.